# BDAPP — Grad-CAM diagnostic pass on Google Colab

This notebook runs the Grad-CAM explainability pass for the BDAPP project, mirroring
`scripts/10_gradcam.py` exactly (same logic, same confusion pairs, same block-average
downsampling), so results are directly comparable to what the local pipeline documents.

**Why this is diagnostic, not a fix.** Work log Sections 15-16: BENIGN leaks into every
attack class because BENIGN has by far the lowest class weight -- but that doesn't say
*which features* actually drive the CNN to mistake a given BENIGN flow for Bot, Web Attack,
or PortScan. Grad-CAM answers that, so any later class-weight/feature fix is evidence-based,
not a guess. This notebook does **not** retrain or change the model -- it only loads the
already-trained checkpoint and inspects it.

**Why Colab.** The real trained checkpoint (epoch 28, the one Section 15's results are
based on) was saved to Google Drive during training (see Section 14) -- it was never
downloaded back to the local machine. Running here loads it directly from Drive instead of
requiring a manual copy. GPU is **optional** for this notebook (Grad-CAM here is a
forward+backward pass over a handful of samples, not training epochs), but using one if
available costs nothing extra.

## 1. Check GPU (optional)

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU -- fine for this notebook, Grad-CAM here is cheap (a handful of samples, not training epochs).")


## 2. Mount Google Drive (same folder the CNN training notebook wrote to)

You'll get a popup asking you to authorize access -- approve it. This reads the checkpoint
and training outputs from `My Drive/BDAPP_cnn_results/`, the same folder
`BDAPP_train_cnn_colab.ipynb` saved to.

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/BDAPP_cnn_results"
MODEL_PATH = os.path.join(DRIVE_OUTPUT_DIR, "cnn_resnet18_best.pt")

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"{MODEL_PATH} not found. This notebook needs the checkpoint saved by "
        "BDAPP_train_cnn_colab.ipynb -- run that first, or fix DRIVE_OUTPUT_DIR."
    )
print("Found checkpoint:", MODEL_PATH)


## 3. Load the encoded test data (via Drive -- not files.upload())

Same reasoning as the training notebook: `files.upload()`/`download()` don't work reliably
from the VS Code Colab extension. Reuses the same `flow_images.zip` you already uploaded for
training (Section 3 there) -- if it's not still on Drive, re-upload it the same way.

In [ ]:
# Adjust this to wherever you uploaded flow_images.zip in your Google Drive
# (same path used in BDAPP_train_cnn_colab.ipynb).
DRIVE_ZIP_PATH = "/content/drive/MyDrive/BDAPP_data/flow_images.zip"

if not os.path.exists(DRIVE_ZIP_PATH):
    raise FileNotFoundError(f"{DRIVE_ZIP_PATH} not found. Upload flow_images.zip to your Google Drive first.")

import zipfile
with zipfile.ZipFile(DRIVE_ZIP_PATH, "r") as zf:
    zf.extractall(".")

DATA_DIR = "flow_images" if os.path.isdir("flow_images") else "."
print("Data dir:", DATA_DIR)
print(os.listdir(DATA_DIR))


## 4. FlowImageDataset (identical to `scripts/flow_image_dataset.py`)

In [ ]:
import cv2
import numpy as np
from torch.utils.data import Dataset

CNN_INPUT_SIZE = 224

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(3, 1, 1)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(3, 1, 1)


class FlowImageDataset(Dataset):
    """Wraps a (N, grid_size, grid_size) uint8 array of flow-image grids and a
    (N,) array of integer labels. Each sample is upscaled to cnn_input_size x
    cnn_input_size with NEAREST-NEIGHBOR interpolation only (never bilinear/
    bicubic -- see the project's Notion Work log, Section 13, for why) and
    normalized the same way the CNN was trained."""

    def __init__(self, grids, labels, cnn_input_size=224):
        self.grids = grids
        self.labels = labels
        self.cnn_input_size = cnn_input_size

    def __len__(self):
        return len(self.grids)

    def __getitem__(self, idx):
        grid = self.grids[idx]
        upscaled = cv2.resize(grid, (self.cnn_input_size, self.cnn_input_size), interpolation=cv2.INTER_NEAREST)
        img = upscaled.astype(np.float32) / 255.0
        img = np.repeat(img[np.newaxis, :, :], 3, axis=0)
        img = (img - IMAGENET_MEAN) / IMAGENET_STD
        return torch.from_numpy(img), int(self.labels[idx])


## 5. Load the checkpoint, encoding metadata, and test set

In [ ]:
import joblib
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

metadata = joblib.load(os.path.join(DATA_DIR, "encoding_metadata.joblib"))
label_encoder = metadata["label_encoder"]
class_names = list(label_encoder.classes_)
feature_order = metadata["feature_order"]
grid_size = metadata["grid_size"]
n_features = metadata["n_features"]
print(f"{len(class_names)} classes, grid_size={grid_size}, n_features={n_features}")

test_grids = np.load(os.path.join(DATA_DIR, "test_grids.npy"))
test_labels = np.load(os.path.join(DATA_DIR, "test_labels.npy"))
print(f"test: {test_grids.shape[0]:,} rows")


def build_model(n_classes):
    model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, n_classes)
    return model


checkpoint = torch.load(MODEL_PATH, map_location=device)
model = build_model(len(class_names)).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Loaded checkpoint: epoch {checkpoint['epoch']}, val macro-F1={checkpoint['val_macro_f1']:.4f}")


## 6. Run inference over the test set (to find the REAL misclassified flows)

The confusion-matrix counts in the Work log (Section 15) are already known, but Grad-CAM
needs the actual sample indices that were misclassified, not just the aggregate counts.

In [ ]:
from torch.utils.data import DataLoader

test_ds = FlowImageDataset(test_grids, test_labels, CNN_INPUT_SIZE)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)

test_pred = []
with torch.no_grad():
    for images, _ in test_loader:
        images = images.to(device)
        test_pred.extend(model(images).argmax(dim=1).cpu().numpy().tolist())
test_pred = np.array(test_pred)

print(f"Test accuracy: {(test_pred == test_labels).mean():.4f}  (sanity check -- should match Section 15's 0.97)")


## 7. Grad-CAM on the three documented confusion pairs

Hooks `model.layer4[-1]` (ResNet18's last conv block), targeting the **predicted** (wrong)
class -- the question is "what made the model say Bot", not "what BENIGN normally looks
like". The 224x224 heatmap is downsampled back to the original 8x8 grid by block-averaging
each exact 28x28 block (224/8=28, no remainder) into one importance score per feature cell
-- never by resizing, which would blur importance across cell boundaries that mean nothing
in a nearest-neighbor-upscaled image.

In [ ]:
!pip install -q grad-cam

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

CONFUSION_PAIRS = [("BENIGN", "Bot"), ("BENIGN", "Web Attack"), ("BENIGN", "PortScan")]
SAMPLES_PER_CONFUSION = 25
TOP_N_FEATURES = 10
RANDOM_STATE = 42

cam = GradCAM(model=model, target_layers=[model.layer4[-1]])


def block_average_to_grid(cam_224, grid_size):
    block = CNN_INPUT_SIZE // grid_size
    return cam_224.reshape(grid_size, block, grid_size, block).mean(axis=(1, 3))


## 8. Run Grad-CAM per pair, rank driving features, save heatmaps + summary CSV

Saves everything to Drive (`DRIVE_OUTPUT_DIR`), the same place the training notebook wrote
to -- nothing here depends on Colab's ephemeral `/content/` surviving a disconnect.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

GRADCAM_HEATMAP_DIR = os.path.join(DRIVE_OUTPUT_DIR, "gradcam_heatmaps")
GRADCAM_SUMMARY_CSV = os.path.join(DRIVE_OUTPUT_DIR, "gradcam_feature_importance_summary.csv")
os.makedirs(GRADCAM_HEATMAP_DIR, exist_ok=True)

rng = np.random.RandomState(RANDOM_STATE)
summary_rows = []

for true_name, pred_name in CONFUSION_PAIRS:
    print(f"\n{'=' * 70}\n{true_name} -> {pred_name}\n{'=' * 70}")

    true_idx = label_encoder.transform([true_name])[0]
    pred_idx = label_encoder.transform([pred_name])[0]
    match_indices = np.where((test_labels == true_idx) & (test_pred == pred_idx))[0]
    n_found = len(match_indices)
    if n_found == 0:
        print("  No misclassified flows found for this pair -- skipping.")
        continue

    sample_indices = rng.choice(match_indices, size=min(SAMPLES_PER_CONFUSION, n_found), replace=False)
    n_used = len(sample_indices)
    print(f"  {n_found} misclassified flows found, averaged over {n_used}.")

    sample_ds = FlowImageDataset(test_grids[sample_indices], test_labels[sample_indices], CNN_INPUT_SIZE)
    batch = torch.stack([sample_ds[i][0] for i in range(len(sample_ds))]).to(device)
    targets = [ClassifierOutputTarget(int(pred_idx))] * n_used
    grayscale_cam = cam(input_tensor=batch, targets=targets)  # (N, 224, 224)

    cell_importances = np.stack([block_average_to_grid(c, grid_size) for c in grayscale_cam])
    mean_importance = cell_importances.mean(axis=0)
    flat_importance = mean_importance.flatten()
    ranked_cells = np.argsort(-flat_importance)

    print(f"  Top {TOP_N_FEATURES} driving features:")
    for rank, cell_idx in enumerate(ranked_cells[:TOP_N_FEATURES], start=1):
        is_padding = cell_idx >= n_features
        feature_name = "<padding cell>" if is_padding else feature_order[cell_idx]
        importance = flat_importance[cell_idx]
        print(f"    {rank:2d}. {feature_name:<30s} importance={importance:.4f}")
        summary_rows.append({
            "confusion_pair": f"{true_name}->{pred_name}",
            "rank": rank,
            "feature": feature_name,
            "importance": importance,
            "n_misclassified_found": n_found,
            "n_samples_used": n_used,
        })

    example_grid = test_grids[sample_indices[0]]
    example_upscaled = cv2.resize(example_grid, (CNN_INPUT_SIZE, CNN_INPUT_SIZE), interpolation=cv2.INTER_NEAREST)
    example_rgb = np.repeat(example_upscaled[:, :, None], 3, axis=2).astype(np.float32) / 255.0
    overlay = show_cam_on_image(example_rgb, grayscale_cam[0], use_rgb=True)

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(overlay)
    ax.set_title(f"{true_name} misclassified as {pred_name}\n(example, Grad-CAM overlay)")
    ax.axis("off")
    overlay_path = os.path.join(GRADCAM_HEATMAP_DIR, f"gradcam_{true_name}_to_{pred_name}.png".replace(" ", "_"))
    fig.savefig(overlay_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"  Saved: {overlay_path}")

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(GRADCAM_SUMMARY_CSV, index=False)
print(f"\nFull summary saved to: {GRADCAM_SUMMARY_CSV}")
summary_df


## 9. Where your results are

Everything was saved to `My Drive/BDAPP_cnn_results/` -- open that folder in Drive directly.

Put the files back into your local project at:

- `gradcam_feature_importance_summary.csv` -> `data/processed/reports/`
- `gradcam_heatmaps/*.png` -> `data/processed/reports/gradcam_heatmaps/`

Then paste the printed per-pair rankings (or the summary table above) back into the chat so
the Work log can be updated with the actual evidence, same as Sections 15 and 17 were.